# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file.
output_file = "./results/toy_problem_uq.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
import numpy as np
from multiprocessing import Process
from args import *

MAX_PROCESSES = 30

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(4)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(10)
default_args["episode_length"] = wrap_arg_val(200)
default_args["seed"] = wrap_arg_val(1)

default_args["observation_mask"] = wrap_arg_val(False)
default_args["random_start_positions"] = wrap_arg_val(True)
default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)
default_args["prediction_during_training"] = wrap_arg_val(False)
default_args["eval_prediction"] = wrap_arg_val(True)

# Evaluation

In [3]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")
    # print("Output file already exists. Overwriting evaluation.")

else:
# if True:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)
    
    obs_probabilities = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

    # Add diffuser series.
    for obs_prob in obs_probabilities:
        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-estimate-draft1/wandb/run-20260513_220746-7w95bqua/files", default_args)
        args["eval_series"] = "Estimate"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-draft1/wandb/run-20260513_220732-tv5hrfgb/files", default_args)
        args["eval_series"] = "Ensemble"
        args["experiment_name"] = wrap_arg_val(f"obs_prob_{obs_prob:.2f}")
        args["observation_probability"] = wrap_arg_val(obs_prob)
        start_process(args)
    
    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--adversary_dynamics', 'random', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--critic_hidden_size', '128', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministic', '--cuda_idx', '4', '--cuda_idx_predictor', '4', '--data_chunk_length', '10', '--no-dec_actor', '--diffusion_autoregression_steps', '1', '--diffusion_epoch', '5', '--diffusion_model_type', 'dit1d', '--diffusion_sample_steps', '20', '--no-disable_observation_global', '--disturbance_drift_magnitude', '0', '--no-encode_state', '--entropy_coef', '0.01', '--env_class', 'toy_proble

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--no-disable_observation_global', '--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.

u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--no-disable_observation_global', '--episode_max', '200', '--no-gnn_neighbor_scoring', '--no-gnn_skip_connections', '--no-observation_mask', '--no-save_checkpoints', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy', '--no-use_gnn_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 4


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be Falseu are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False

Pettingzoo environment toy_problem.toy_problem_v1.parallel_env has additional arguments.


...

R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: FalseAction output distributions:

  0: DiagGaussian, action_dim: 2, available_action_dim: inf
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: FalseR_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False

Action output distributions:
Action output distributions:  0: DiagGaussian, action_dim: 2, available_action_dim: inf

  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action output distributions:
  0: DiagGaussian, action_dim: 2, available_action_dim: inf
R_Actor: Use GNN: False, Use CNN: False, Use MLP: True, Use RNN: False, Use Encoder: False
Action ou

# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='Ensemble', nodes=(Node(disabled=Tru…

In [5]:
%reload_ext autoreload
%autoreload 2
from plots import *
import numpy as np

x_key = 'observation_probability'

# Data for plotting
plots = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][experiment]['tracking_error_mean']) for experiment in output[series]]
                for series in output
        }
    },
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        },
        'y_std': {
            series: [np.std(np.mean(output[series][exp]['prediction_error_mean'], axis=tuple(range(1, output[series][exp]['prediction_error_mean'].ndim)))) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_min': {
        'title': 'Prediction Uncertainty Gap (Min)',
        'ylabel': 'Prediction Uncertainty Gap Min (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_min']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    },
    'prediction_uncertainty_gap_max': {
        'title': 'Prediction Uncertainty Gap (Max)',
        'ylabel': 'Prediction Uncertainty Gap Max (units distance)',
        'xlabel': 'Observation Probability',
        'x': {series: get_x_radii(series, output, x_key) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_max']) for exp in sorted_experiments(series, output, x_key)]
                for series in output if not next(iter(output[series].values())).attrs['prediction_disable']
        }
    }
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

plot(plots, timesteps=timesteps)
